# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdullahhashmi01/FlyRank-ML-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

from google.colab import files

uploaded = files.upload()

file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name)

print("Dataset loaded successfully")
print("Rows:", len(df))
print("Columns:", len(df.columns))


Saving content_refresh_anonymized.csv to content_refresh_anonymized (2).csv
Dataset loaded successfully
Rows: 30000
Columns: 44


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
paper_review = pd.DataFrame({
    "finding": [
        "Growing content is longer and younger",
        "Growth model achieved 71% accuracy"
    ],
    "paper_result": [
        "3180 vs 2311 words; 184 vs 230 days",
        "71% holdout accuracy"
    ],
    "label_source": [
        "30-day vs previous-30-day impression trend",
        "Rule-defined growth direction"
    ],
    "methodology_question": [
        "Does the pattern remain within similar clients and content types?",
        "Does the score remain similar with a grouped-client split?"
    ]
})

display(paper_review)

,finding,paper_result,label_source,methodology_question
0,Growing content is longer and younger,3180 vs 2311 words; 184 vs 230 days,30-day vs previous-30-day impression trend,Does the pattern remain within similar clients...
1,Growth model achieved 71% accuracy,71% holdout accuracy,Rule-defined growth direction,Does the score remain similar with a grouped-c...


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [27]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    GroupShuffleSplit
)
from sklearn.ensemble import RandomForestClassifier


# ── 1. Safe numeric features select karo ──────────────────────────────────────
feature_candidates = [
    "clicks",
    "impressions",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "word_count"
]

final_features = [
    col for col in feature_candidates
    if col in df.columns
]

print("Features used:", final_features)


# ── 2. Numeric feature dataframe banao ────────────────────────────────────────
X = df[final_features].copy()

for col in final_features:
    X[col] = pd.to_numeric(X[col], errors="coerce")


# ── 3. avg_position fix karo (0 → NaN) ───────────────────────────────────────
if "avg_position" in X.columns:
    X["avg_position"] = X["avg_position"].replace(0, np.nan)


# ── 4. Target prepare karo ────────────────────────────────────────────────────
target_column = "trend_direction"

target_text = (
    df[target_column]
    .astype(str)
    .str.strip()
    .str.lower()
)

target_map = {
    "1":            1,
    "1.0":          1,
    "true":         1,
    "yes":          1,
    "declining":    1,
    "down":         1,
    "0":            0,
    "0.0":          0,
    "false":        0,
    "no":           0,
    "not_declining": 0,
    "up":           0,
    "stable":       0,
}

y = target_text.map(target_map)


# ── 5. Sirf valid rows rakho ──────────────────────────────────────────────────
valid_rows = y.notna() & df["client_id"].notna()

X       = X.loc[valid_rows].reset_index(drop=True)
y       = y.loc[valid_rows].astype(int).reset_index(drop=True)
groups  = df.loc[valid_rows, "client_id"].reset_index(drop=True)

print("Rows used:        ", len(X))
print("Declining rows:   ", int(y.sum()))
print("Overall base rate:", round(y.mean(), 3))


# ── 6. Split indices banao ────────────────────────────────────────────────────
indices = np.arange(len(X))

# Random split
random_train, random_test = train_test_split(
    indices,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Grouped split (client overlap nahi hoga)
group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

group_train, group_test = next(
    group_splitter.split(X, y, groups)
)


# ── 7. Missing values fill karo (sirf train medians se) ──────────────────────
def prepare_split(train_indices, test_indices):
    X_train = X.iloc[train_indices].copy()
    X_test  = X.iloc[test_indices].copy()

    # Medians sirf training data se liye jayein
    train_medians = X_train.median()

    X_train = X_train.fillna(train_medians).fillna(0)
    X_test  = X_test.fillna(train_medians).fillna(0)

    return X_train, X_test


# ── 8. Splits prepare karo ───────────────────────────────────────────────────
X_random_train, X_random_test = prepare_split(random_train, random_test)
X_group_train,  X_group_test  = prepare_split(group_train,  group_test)


# ── 9. Model banao ───────────────────────────────────────────────────────────
model_params = dict(
    n_estimators=150,
    max_depth=12,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

random_model  = RandomForestClassifier(**model_params)
grouped_model = RandomForestClassifier(**model_params)


# ── 10. Train karo ───────────────────────────────────────────────────────────
random_model.fit(X_random_train, y.iloc[random_train])
grouped_model.fit(X_group_train, y.iloc[group_train])


# ── 11. Decline-risk scores nikalo ───────────────────────────────────────────
random_scores  = random_model.predict_proba(X_random_test)[:, 1]
grouped_scores = grouped_model.predict_proba(X_group_test)[:, 1]


# ── 12. Precision@20 function ────────────────────────────────────────────────
def precision_at_20(y_true, scores):
    result = pd.DataFrame({
        "actual": np.asarray(y_true),
        "score":  scores
    })
    top_20 = result.sort_values("score", ascending=False).head(20)
    return top_20["actual"].mean()


# ── 13. Results calculate karo ───────────────────────────────────────────────
random_precision  = precision_at_20(y.iloc[random_test],  random_scores)
grouped_precision = precision_at_20(y.iloc[group_test],   grouped_scores)

random_base_rate  = y.iloc[random_test].mean()
grouped_base_rate = y.iloc[group_test].mean()


# ── 14. Client overlap check karo ────────────────────────────────────────────
train_clients  = set(groups.iloc[group_train])
test_clients   = set(groups.iloc[group_test])
client_overlap = len(train_clients.intersection(test_clients))


# ── 15. Results table ─────────────────────────────────────────────────────────
validation_results = pd.DataFrame({
    "split": [
        "Before: random row split",
        "After:  grouped-client split"
    ],
    "base_rate": [
        round(random_base_rate,  3),
        round(grouped_base_rate, 3)
    ],
    "precision_at_20": [
        round(random_precision,  3),
        round(grouped_precision, 3)
    ],
    "client_overlap": [
        "Possible (random split)",
        client_overlap           # grouped split mein 0 hona chahiye
    ]
})

display(validation_results)

Features used: ['ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'word_count']
Rows used:         26612
Declining rows:    16262
Overall base rate: 0.611


,split,base_rate,precision_at_20,client_overlap
0,Before: random row split,0.611,0.90,Possible (random split)
1,After: grouped-client split,0.617,0.65,0


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [28]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Fields that must never appear in the model
blocked_fields = [
    "is_declining_label",
    "trend_direction",
    "trend_pct",
    "content_id",
    "client_id",
    "health_score",
    "priority_score",
    "optimization_flag"
]

# Check whether any blocked field was used
blocked_fields_used = []

for field in final_features:
    if field in blocked_fields:
        blocked_fields_used.append(field)


# Check suspicious words
blocked_words = [
    "label",
    "target",
    "future",
    "outcome",
    "trend",
    "health",
    "priority",
    "query",
    "url"
]

suspicious_fields_used = []

for field in final_features:
    for word in blocked_words:
        if word in field.lower():
            suspicious_fields_used.append(field)
            break


# Make an easy audit table
leakage_audit = pd.DataFrame({
    "audit_check": [
        "Label used as a feature",
        "trend_direction used",
        "trend_pct used",
        "Client ID used as a feature",
        "Content ID used as a feature",
        "Private query or URL used",
        "Product score or flag used",
        "Client overlap in grouped split",
        "Feature and label windows overlap"
    ],
    "result": [
        target_column in final_features,
        "trend_direction" in final_features,
        "trend_pct" in final_features,
        "client_id" in final_features,
        "content_id" in final_features,
        len(suspicious_fields_used) > 0,
        any(
            word in field.lower()
            for field in final_features
            for word in [
                "health",
                "priority",
                "optimization"
            ]
        ),
        client_overlap > 0,
        True
    ],
    "interpretation": [
        "Must be False",
        "Must be False",
        "Must be False",
        "Must be False",
        "Must be False",
        "Must be False",
        "Must be False",
        "Must be False",
        "Known limitation of the starter snapshot"
    ]
})

print("Final features:")
print(final_features)

print("\nBlocked fields used:")
print(blocked_fields_used)

print("\nSuspicious fields used:")
print(suspicious_fields_used)

display(leakage_audit)


Final features:
['ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'word_count']

Blocked fields used:
[]

Suspicious fields used:
[]


,audit_check,result,interpretation
0,Label used as a feature,False,Must be False
1,trend_direction used,False,Must be False
2,trend_pct used,False,Must be False
3,Client ID used as a feature,False,Must be False
4,Content ID used as a feature,False,Must be False
5,Private query or URL used,False,Must be False
6,Product score or flag used,False,Must be False
7,Client overlap in grouped split,False,Must be False
8,Feature and label windows overlap,True,Known limitation of the starter snapshot


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.